## Final Project
# **RegTech Platform for Regulatory Surveillance**

#### **STEP 1. Data Ingestion and Extraction (Data Collection)**
- Explanation: Creation of the modules responsible for visiting the EMA and AEMPS websites to extract regulatory texts. It will be designed with an object-oriented approach, creating independent extractors for each agency that handle exceptions and network failures gracefully.
- Technologies: Python, scraping libraries like BeautifulSoup , and Requests.

In [10]:
import requests
from bs4 import BeautifulSoup
import time


def scrape_ema_multiple_baselines(url_list):
    """
    Iterates over a list of dictionaries containing URLs and target filenames.
    Scrapes each EMA page to establish multiple Scenario 1 (Baseline) files.
    Includes a polite delay between requests to prevent IP blocking.
    """
    # Define request headers to simulate a real browser session
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    counter=0
    for item in url_list:
        target_url = item.get("url")
        output_filename = item.get("filename")
        counter += 1
        try:
            print(f"Connecting to EMA target URL: {target_url}...")
            response = requests.get(target_url, headers=headers, timeout=15)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Extract the main document title
                title_element = soup.find('h1')
                title_text = title_element.get_text(strip=True) if title_element else "No Title Found"
                
                # Extract semantic text paragraphs (ignoring short UI/menu elements)
                paragraphs = [p.get_text(strip=True) for p in soup.find_all('p') if len(p.get_text(strip=True)) > 20]
                
                # Track links and attached documents to detect additions/deletions of PDFs
                tracked_links = []
                for anchor in soup.find_all('a', href=True):
                    anchor_text = anchor.get_text(strip=True)
                    href = anchor['href']
                    
                    # Filter for meaningful anchor texts or explicit document extensions
                    is_document = any(ext in href.lower() for ext in ['.pdf', '.doc', '.docx', '.xlsx'])
                    if len(anchor_text) > 10 or is_document:
                        tracked_links.append(f"LINK_TEXT: {anchor_text} | TARGET_URL: {href}")
                
                # Consolidate all extracted data into a single structured string
                consolidated_content = f"TITLE: {title_text}\n\n"
                consolidated_content += "=== TEXT CONTENT ===\n" + "\n\n".join(paragraphs) + "\n\n"
                consolidated_content += "=== ATTACHMENTS AND TRACKED LINKS ===\n" + "\n".join(tracked_links)
                
                # Persist data locally as the ground truth baseline
                with open(output_filename, "w", encoding="utf-8") as file:
                    file.write(consolidated_content)
                                       
                print(f"Success! Data saved to: '{output_filename}'\n")
                
            else:
                print(f"Access denied for {target_url}. HTTP Status Code: {response.status_code}\n")
                
        except Exception as error:
            print(f"An error occurred while processing {target_url}: {error}\n")
            
        # Polite scraping delay to avoid server overload or rate limiting
        print("Waiting 2 seconds before the next request...")
        time.sleep(2) 
        
    print(f"All URLs have been processed successfully. Total processed: {counter}")


In [11]:
# --- Execution Block ---
import os

if __name__ == "__main__":
    
    # 1. Folder & version
    output_dir = "Data_Extraction"
    version = "v1.2_"
    
    # 2. Create the folder at the same level as the notebook (exist_ok=True prevents errors if it exists)
    os.makedirs(output_dir, exist_ok=True)
    
    # 3. Define the URLs and build the file paths safely
    ema_target_sections = [
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview", "filename": os.path.join(output_dir, version + "ema_scenario_1_regulatory.txt")},
        {"url": "https://www.ema.europa.eu/en/news-events", "filename": os.path.join(output_dir, version + "ema_scenario_1_news.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_reg_research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/advanced-therapy-medicinal-products-overview", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_advanced-therapy-medicinal-products-overview.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/cancer-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_cancer-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/clinical-trials-human-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_clinical-trials-human-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-laboratory-practice-compliance", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-laboratory-practice-compliance.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/compliance-research-development/good-manufacturing-practice", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_compliance-research-development_good-manufacturing-practice.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/compliance-post-authorisation/good-distribution-practice", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_post-authorisation_compliance-post-authorisation_good-distribution-practice.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/quality-design", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_quality-design.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A50", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Compliance_inspections.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A56", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Regulatory_procedural_guidance.txt")},
        {"url": "https://www.ema.europa.eu/en/search?f%5B0%5D=ema_search_topics%3A72", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_Research_development.txt")},
        {"url": "https://www.ema.europa.eu/en/data-medicines-iso-idmp-standards-research-development", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_data-medicines-iso-idmp-standards-research-development.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/prime-priority-medicines", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_research-development_prime-priority-medicines.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/research-development/scientific-advice-protocol-assistance", "filename": os.path.join(output_dir, version + "ema_scenario_1_research-development_scientific-advice-protocol-assistance.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/marketing-authorisation", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_marketing-authorisation.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_post-authorisation.txt")},
        {"url": "https://www.ema.europa.eu/en/human-regulatory-overview/post-authorisation/certification-medicinal-products", "filename": os.path.join(output_dir, version + "ema_scenario_1_post-authorisation_certification-medicinal-products.txt")},
        {"url": "https://health.ec.europa.eu/medicinal-products/eudralex_en", "filename": os.path.join(output_dir, version + "ema_scenario_1_Hum_medicinal-products_eudralex_en.txt")}
    ] 
   
    # ==========================================
    # PHASE 1: DATA SCRAPING
    # ==========================================
    print("\n=== STARTING DATA EXTRACTION (SCRAPING) ===")
    # Call your intact scraping function
    scrape_ema_multiple_baselines(ema_target_sections)
    


=== STARTING DATA EXTRACTION (SCRAPING) ===
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_regulatory.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/news-events...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_news.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview/research-development...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_Hum_reg_research-development.txt'

Waiting 2 seconds before the next request...
Connecting to EMA target URL: https://www.ema.europa.eu/en/human-regulatory-overview/advanced-therapy-medicinal-products-overview...
Success! Data saved to: 'Data_Extraction\v1.2_ema_scenario_1_Hum_advanced-therapy-medicinal-products-overview.txt'

Waiting 2 seconds before the next request...
Connectin

#### **STEP 2. Historical Storage and Versioning (Database)**
- Explanation: Design of a centralized repository to store the raw extracted texts along with their metadata (agency, extraction date, link). This layer will ensure the temporal traceability needed to compare the "before" and "after" in on-demand queries.
- Technologies: A relational database approach was chosen, implemented via SQLite and managed through the SQLAlchemy ORM in Python

In [12]:
import os

from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker
from datetime import datetime


# ==========================================
# DATABASE SCHEMA SETUP
# ==========================================
Base = declarative_base()

class RegulatorySnapshot(Base):
    __tablename__ = 'regulatory_snapshots'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    agency = Column(String(50), nullable=False)
    url = Column(String(255), nullable=False)
    extraction_date = Column(DateTime, default=datetime.utcnow)
    title = Column(String(255))
    text_content = Column(Text, nullable=False)
    tracked_links = Column(Text)

os.makedirs('BD', exist_ok=True)
engine = create_engine('sqlite:///BD/regtech_data.db', echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)

# ==========================================
# FUNCTIONS
# ==========================================
def save_snapshot_to_db(session, agency, url, title, text_content, tracked_links):
    """Saves a new scraped regulatory snapshot into the database."""
    try:
        new_snapshot = RegulatorySnapshot(
            agency=agency,
            url=url,
            title=title,
            text_content=text_content,
            tracked_links=tracked_links
        )
        session.add(new_snapshot)
        session.commit()
        return True
    except Exception as e:
        session.rollback()
        print(f"  [!] Failed to save to database: {e}")
        return False

def parse_scraped_file(filepath):
    """Reads the local .txt file and splits the Title, Content, and Links."""
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    title, text_content, tracked_links = "No Title", "", ""

    try:
        if "TITLE:" in content:
            title = content.split("=== TEXT CONTENT ===")[0].replace("TITLE:", "").strip()
        if "=== TEXT CONTENT ===" in content and "=== ATTACHMENTS AND TRACKED LINKS ===" in content:
            text_content = content.split("=== TEXT CONTENT ===")[1].split("=== ATTACHMENTS AND TRACKED LINKS ===")[0].strip()
        if "=== ATTACHMENTS AND TRACKED LINKS ===" in content:
            tracked_links = content.split("=== ATTACHMENTS AND TRACKED LINKS ===")[1].strip()
    except Exception as e:
        print(f"  [!] Error parsing structure in {filepath}: {e}")

    return title, text_content, tracked_links

# ==========================================
#  DATABASE LOADING
# ==========================================
print("\n=== STARTING SQLITE DATABASE LOAD ===")
    
db_session = Session()
uploaded_files = 0
failed_files = []
    
for item in ema_target_sections:
    filepath = item["filename"]
    target_url = item["url"]
    filename_only = os.path.basename(filepath)
        
    if os.path.exists(filepath):
        parsed_title, parsed_text, parsed_links = parse_scraped_file(filepath)
            
        # Syntax fixed: No line break before parentheses
        success = save_snapshot_to_db(
                session=db_session,
                agency="EMA",
                url=target_url,
                title=parsed_title,
                text_content=parsed_text,
                tracked_links=parsed_links
            )
            
        if success:
            uploaded_files += 1
        else:
            failed_files.append(f"{filename_only} (Failed to insert into DB)")
    else:
        failed_files.append(f"{filename_only} (File not found. Possible extraction failure.)")
            
db_session.close()
    
# ==========================================
# PHASE 3: FINAL REPORT
# ==========================================
print("\n" + "="*50)
print("📋 FINAL PIPELINE REPORT")
print("="*50)
print(f"✅ Total URLs processed and saved to DB: {uploaded_files} out of {len(ema_target_sections)}")
    
if len(failed_files) > 0:
    print(f"❌ Files NOT processed ({len(failed_files)}):")
    for failure in failed_files:
        print(f"   - {failure}")
    else:
        print("🎉 End-to-End Pipeline completed without errors! All texts are in SQLite.")
print("="*50)


=== STARTING SQLITE DATABASE LOAD ===

📋 FINAL PIPELINE REPORT
✅ Total URLs processed and saved to DB: 21 out of 21
